# GAN Urban Design v2 — External Modeller (CycleGAN, Pix2PixHD)

Bu notebook, **`colab_pix2pix.ipynb`'in 5-6. hücreleri (veri indirme + sentetik sketch üretimi) önceden çalıştırılmış** varsayar. Eksikse önce o notebook'taki Hücre 1-6'yı koşturun, sonra buraya gelin.

**Çalıştırma sırası:**
1. Drive bağla + repo klonla + bağımlılıklar
2. External repoları çek (`setup_external.sh`)
3. Veriyi external repoların beklediği formata uyumlulaştır
4. CycleGAN eğit + test + değerlendir (~2 saat A100)
5. Pix2PixHD eğit + test + değerlendir (~4-5 saat A100)
6. Sonuçları tek tabloda topla

**Toplam süre:** ~7 saat A100. Sırayla başlatın, ardı ardına çalışsınlar.

## 1) Ortam kurulumu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, pathlib
GITHUB_USER = 'ilker-23'
REPO = 'gan-urban-design-v2'

os.chdir('/content')
if not os.path.isdir(REPO):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git
else:
    !git -C {REPO} pull --ff-only
os.chdir(f'/content/{REPO}')

!pip install -q -r requirements.txt
# CycleGAN/Pix2PixHD'nin ekstra bağımlılıkları
!pip install -q dominate visdom

!nvidia-smi | head -10

## 2) Veriyi kontrol et (önceden hazırlanmış olmalı)
Eğer aşağıdaki ls boşsa, `colab_pix2pix.ipynb` Hücre 5-6'yı önce çalıştırın.

In [ ]:
!ls data/processed/maps_sketch/train/A 2>/dev/null | wc -l
!ls data/processed/maps_sketch/val/A 2>/dev/null | wc -l
# Beklenen: 1096 ve 1098

## 3) External repoları çek

In [ ]:
!bash scripts/setup_external.sh
!ls external/

## 4) Veriyi external formatlara uyumlulaştır

- `data/external/junyanz_cyc/{trainA,trainB,testA,testB}/` — CycleGAN (unaligned)
- `data/external/junyanz_pix/{train,test}/` — Pix2Pix aligned (AB yan yana)
- `data/external/pix2pixhd/{train_A,train_B,test_A,test_B}/` — Pix2PixHD

Symlink kullanır, ek disk kullanmaz.

In [ ]:
!python scripts/prepare_external_data.py --src data/processed/maps_sketch --out data/external

In [ ]:
# Doğrula
!ls data/external/junyanz_cyc/trainA | wc -l
!ls data/external/junyanz_cyc/trainB | wc -l
!ls data/external/pix2pixhd/train_A | wc -l
!ls data/external/pix2pixhd/train_B | wc -l

## 5) CycleGAN eğitimi

**Süre:** ~2 saat A100 (256², 200 epoch).

Checkpoint'ler `external/.../checkpoints/sketch2map_cyc/` altına yazılır. Drive'a kopyalama hücresi en sonda var.

In [ ]:
%cd /content/{REPO}/external/pytorch-CycleGAN-and-pix2pix
!python train.py \
    --dataroot /content/{REPO}/data/external/junyanz_cyc \
    --name sketch2map_cyc \
    --model cycle_gan \
    --load_size 286 --crop_size 256 \
    --batch_size 4 \
    --n_epochs 100 --n_epochs_decay 100 \
    --save_epoch_freq 25 \
    --display_id 0
%cd /content/{REPO}

## 6) CycleGAN testi (val kümesinde sketch → map çevirisi)

In [ ]:
%cd /content/{REPO}/external/pytorch-CycleGAN-and-pix2pix
!python test.py \
    --dataroot /content/{REPO}/data/external/junyanz_cyc \
    --name sketch2map_cyc \
    --model cycle_gan \
    --load_size 256 --crop_size 256 \
    --num_test 1098 \
    --no_dropout
%cd /content/{REPO}
!ls external/pytorch-CycleGAN-and-pix2pix/results/sketch2map_cyc/test_latest/images | head -5

## 7) CycleGAN değerlendirmesi (FID, SSIM, PSNR, LPIPS, L1)

In [ ]:
!python scripts/evaluate_external.py \
    --results-dir external/pytorch-CycleGAN-and-pix2pix/results/sketch2map_cyc/test_latest/images \
    --pattern junyanz \
    --tag cyclegan_sketch2map

## 8) Drive'a yedek (CycleGAN bittikten sonra)

In [ ]:
!mkdir -p /content/drive/MyDrive/thesis_runs/cyclegan_sketch2map
!cp -r external/pytorch-CycleGAN-and-pix2pix/checkpoints/sketch2map_cyc \
      /content/drive/MyDrive/thesis_runs/cyclegan_sketch2map/checkpoints
!cp -r results/external_eval/cyclegan_sketch2map \
      /content/drive/MyDrive/thesis_runs/cyclegan_sketch2map/eval
!ls /content/drive/MyDrive/thesis_runs/cyclegan_sketch2map/

---
## 9) Pix2PixHD eğitimi (512² × 150 epoch ≈ 4-5 saat)

**Önemli:** NVIDIA pix2pixHD repo'su, görüntü çiftlerinden çevirim için `--label_nc 0 --no_instance` modunu kullanır. Bu modda `train_A` girdi (sketch), `train_B` hedef (map) olarak çalışır.

In [ ]:
%cd /content/{REPO}/external/pix2pixHD
!python train.py \
    --name sketch2map_hd \
    --dataroot /content/{REPO}/data/external/pix2pixhd \
    --label_nc 0 --no_instance \
    --resize_or_crop scale_width \
    --loadSize 512 --fineSize 512 \
    --batchSize 2 \
    --niter 100 --niter_decay 50 \
    --save_epoch_freq 25 \
    --display_id 0
%cd /content/{REPO}

## 10) Pix2PixHD testi

In [ ]:
%cd /content/{REPO}/external/pix2pixHD
!python test.py \
    --name sketch2map_hd \
    --dataroot /content/{REPO}/data/external/pix2pixhd \
    --label_nc 0 --no_instance \
    --resize_or_crop scale_width \
    --loadSize 512 --fineSize 512 \
    --how_many 1098
%cd /content/{REPO}

## 11) Pix2PixHD değerlendirmesi

In [ ]:
!python scripts/evaluate_external.py \
    --results-dir external/pix2pixHD/results/sketch2map_hd/test_latest/images \
    --pattern pix2pixhd \
    --tag pix2pixhd_sketch2map

## 12) Pix2PixHD Drive yedeği

In [ ]:
!mkdir -p /content/drive/MyDrive/thesis_runs/pix2pixhd_sketch2map
!cp -r external/pix2pixHD/checkpoints/sketch2map_hd \
      /content/drive/MyDrive/thesis_runs/pix2pixhd_sketch2map/checkpoints
!cp -r results/external_eval/pix2pixhd_sketch2map \
      /content/drive/MyDrive/thesis_runs/pix2pixhd_sketch2map/eval

---
## 13) Tüm sonuçları tek tabloda topla

In [ ]:
import json, pathlib, pandas as pd

runs = {
    'Pix2Pix (bizim)':  '/content/drive/MyDrive/thesis_runs/pix2pix_sketch2map/eval/metrics.json',
    'CycleGAN':         'results/external_eval/cyclegan_sketch2map/metrics.json',
    'Pix2PixHD':        'results/external_eval/pix2pixhd_sketch2map/metrics.json',
}

rows = []
for name, path in runs.items():
    p = pathlib.Path(path)
    if not p.exists():
        rows.append({'Model': name, 'FID': None, 'SSIM': None, 'PSNR': None, 'LPIPS': None, 'L1': None}); continue
    s = json.loads(p.read_text())['summary']
    rows.append({'Model': name,
                 'FID':   round(s.get('fid',         float('nan')), 2),
                 'SSIM':  round(s['ssim_mean'],  4),
                 'PSNR':  round(s['psnr_mean'],  2),
                 'LPIPS': round(s['lpips_mean'], 4),
                 'L1':    round(s['l1_mean'],    2)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('/content/drive/MyDrive/thesis_runs/comparison_table.csv', index=False)
print('\nKaydedildi: /content/drive/MyDrive/thesis_runs/comparison_table.csv')

---
## SPADE (DeepGlobe Semantic → Aerial) — Ayrı Notebook'ta

SPADE'in mantığı `sketch → map` değil `semantic mask → photo` olduğu için, **DeepGlobe Land Cover** veri kümesi gerekir ve farklı bir hazırlık yapılır. Bu deneyi şu an atlayabilir, tez teslim sonrası ek deney olarak değerlendirebilirsiniz.

İlgili komutlar repo'da `notebooks/colab_spade_deepglobe.ipynb` olarak gelecektir (planlanan).